In [7]:
import flopy
import flopy.mt3d
import numpy as np
import pandas as pd
import csv
import matplotlib as mpl
from matplotlib_inline.backend_inline import flush_figures
import matplotlib.pyplot as plt
import flopy.utils.binaryfile as bf
import flopy.utils.postprocessing as post
import ipywidgets as widgets
    
def run_model(hk1, hk2, hk3, specific_yield, riv_conductance, transport):
    #######################################################################################################################################################
    
    # Create a MODFLOW model
    modelname = "model_flow"
    model = flopy.modflow.Modflow(modelname, exe_name="2_mf2005")

    ########################################################################################################################################################

    # Now, add required packages (DIS, BAS) to the model. The model will not work without these packages. 

    ######################################################################################################################################################

    # Define the model's spatial unit and dimensions
    lenuni = 2 # (0-undefined, 1-feet, 2-meters, 3-centimeters)
    Lx = 400.0 # meters
    Ly = 1.0 # meters
    ztop = 48.0 # meters
    zbot = 0.0 # meters

    # Define the model's spatial discretization
    nlay = 12  # number of layers
    ncol = 100  # number of columns
    nrow = 1  # number of rows 
    delr = Lx / ncol  # cell width in x direction (meters)
    delc = Ly / nrow  # cell width in y direction (meters)

    # Define cell width in z-direction to define the bottom elevation of each layer in the model (12 layers). 
    # Layer 1 is at the highest elevation (48m - 44m). Layer 12 is at the lowest elevation (4m - 0m). 
    cell_depth = 4 # cell width in z direction (meters)
    bottom_layer_1 = 44 # meters
    botm = bottom_layer_1 - np.arange(0, cell_depth * nlay, cell_depth)
    botm = botm.reshape(nlay, 1, 1)
    botm = np.repeat(botm, nrow, axis=1)
    botm = np.repeat(botm, ncol, axis=2)  # botm array built without using a for-loop. more efficient. 

    # Define the model temporal dicretization 
    itmuni = 4  # (1-seconds, 2-minutes, 3-hours, 4-days, 5-years)
    nper = 366 # number of stress periods (0-365). must be an integer.
    perlen = np.ones(366, dtype=int) # length of stress period in days. integer or array or list. 
    nstp = 24*np.ones(366, dtype=int) # number of timesteps in a stress period. integer or array or list. 

    # Create an array for the steady state variable in the discretization package
    steady_array = np.full(366, False)
    steady_array[0] = True # Overwrite the first item and set to "True"

    # Add Discretization (DIS) package
    flopy.modflow.ModflowDis(model, lenuni = lenuni, nlay = nlay, nrow = nrow, ncol = ncol, delr = delr, delc = delc, 
                             top = ztop, botm = botm, itmuni = itmuni, nper = nper, perlen = perlen, nstp = nstp, 
                             steady = steady_array)

    ######################################################################################################################################################

    # Define the active/inactive cells, the type of cell (constant head, inactive/no-flow, variable head)
    ibound = np.ones((nlay, nrow, ncol), dtype=np.int32)
    ibound[:, :, -1] = -1 # constant boundary condition for last column
    
    # Define the starting head of each cell
    strt = np.ones((nlay, nrow, ncol), dtype=np.float32)
    strt[:, :, :] =  48.0 # head value in all cells for the first stress period...simply to prevent numerical errors 

    # Add Basic (BAS) Package
    flopy.modflow.ModflowBas(model, ibound=ibound, strt=strt, hnoflo=-1e30)

    ######################################################################################################################################################

    # Now, add non-required packages (LPF, RIV, PCG, OC, etc.) to the model

    ######################################################################################################################################################

    # Define layer properites: type of aquifer (confined/unconfined), hydraulic conductivity for each layer (meters/day), porosity, and specific yield
    laytyp = np.zeros(nlay) # Empty array to store layer type values
    hk  = np.zeros((nlay, nrow, ncol)) # Empty array to store hydraulic conductivity values
    vka = np.zeros((nlay, nrow, ncol)) # Empty array to store vertical hydraulic conductivity values
    sy  = np.zeros((nlay, nrow, ncol)) # Empty array to store specific yield values
    porosity = 0.3 # to be used in transport equations, not required for flow

    # Fill the empty arrays and assign parameter values to the arrays accordingly 
    laytyp[0:12] = 1     # laytyp for all layers. 0 = confined. 1 = convertible. -1 = see modflow or flopy documentation.
    hk[0:4, :, :] = hk1   # Hydraulic conductivity of layers 0-3 for all cells in layer
    hk[4:8, :, :] = hk2    # Hydraulic conductivity of Layers 4-7 for all cells in layer
    hk[8:12, :, :] = hk3 # Hydraulic conductivity of Layers 8-11 for all cells in layer
    vka[0:4, :, :] = hk[0:4, :, :]*0.1    # Vertical Hydraulic conductivity of Layers 0-3 for all cells in layer
    vka[4:8, :, :] = hk[4:8, :, :]*0.1    # Vertical Hydraulic conductivity of Layers 4-7 for all cells in layer
    vka[8:12, :, :] = hk[8:12, :, :]*0.1  # Vertical Hydraulic conductivity of Layers 8-11 for all cells in layer
    sy[:,:,:] = specific_yield # Specific yield, entire model domain

    # Add Layer Property Flow Package
    flopy.modflow.ModflowLpf(model, ipakcb=1, hdry=-1e29, laytyp=laytyp, laywet=1, hk=hk, sy=sy, vka=vka, wetfct=1, ihdwet=1)

    ######################################################################################################################################################

    # Load the river stress period data from "riv_stress_period_data.csv". To work, the csv file must be in the same folder as the code
    csv_filename = "4_riv_stress_period_data.csv"
    river_data_df = pd.read_csv(csv_filename)

    # Initialize an empty dictionary
    riv_stress_period_data = {}

    # Group the data by 'nper' and convert it to the desired dictionary format
    for stress_period, group in river_data_df.groupby('nper'):
        stress_period_data = group[['layer', 'row', 'column', 'stage', 'riv_botm']].values.tolist()
        # Add riv_conductance as the 5th item in the list
        for row_data in stress_period_data:
            row_data.insert(4, riv_conductance)
        # Add the data to the dictionary
        riv_stress_period_data[stress_period] = stress_period_data

    # Now, riv_stress_period_data contains the river stress period data in the required format for the flopy riv package

    # Add RIV Package
    flopy.modflow.ModflowRiv(model, ipakcb=1, stress_period_data=riv_stress_period_data)

    ######################################################################################################################################################

    # Add PCG (the numerical solver) Package
    flopy.modflow.ModflowPcg(model, iter1=100, hclose=1e-3, rclose=1e-3, mutpcg=0)

    ######################################################################################################################################################

    # Define the stress periods for output control using a dictionary of lists
    oc_stress_period_data = {(stressperiod, 23): ['SAVE HEAD', 'PRINT HEAD', 'PRINT BUDGET', 'SAVE BUDGET']
                            for stressperiod in range(nper)}

    # Add Output Control Package
    flopy.modflow.ModflowOc(model, stress_period_data=oc_stress_period_data)

    ######################################################################################################################################################

    # LMT7 (MT3D_Link) Transport Package. This package creates a file that is required for the transport model.
    lmt = flopy.modflow.ModflowLmt(model, output_file_name='mt3d_link.ftl')

    ######################################################################################################################################################

    # Write flow model input files and run the flow model
    model.write_input()
    model.run_model()
    model.check()

    ######################################################################################################################################################
    # Print the model's head results into the "5_head_data_layer5.csv" file

    # Read head data from all stress periods and time steps
    hds = bf.HeadFile(modelname + '.hds')
    all_times = hds.get_times()

    # Extract head data for the specified times and locations in layer 5
    locations = [(4, 0, 2), (4, 0, 12), (4, 0, 24)] 
    head_data_selected = np.array([[hds.get_data(totim=time)[loc] for loc in locations] for time in all_times])

    # Construct a DataFrame for the data
    stress_periods = [f'Stress Period {i}' for i in range(len(all_times))]
    columns = ['Layer 5, Row 1, Column 3', 'Layer 5, Row 1, Column 13', 'Layer 5, Row 1, Column 25']
    df_formatted = pd.DataFrame(head_data_selected, columns=columns)
    df_formatted.insert(0, 'Stress Period', stress_periods)

    # Save the DataFrame to a CSV file
    csv_filename2 = "5_head_data_layer5.csv"
    df_formatted.to_csv(csv_filename2, index=False, header=True)


    ##########################################################################################################################################################
    if transport:
       
        # Initialize the MT3D model.
        modelname = "model_transport"
        mt = flopy.mt3d.Mt3dms(modelname, exe_name='3_mt3dms5b', modflowmodel=model, ftlfilename="mt3d_link.ftl", verbose = True)

        ######################################################################################################################################################

        # Concentration units can cause stability issues. If isssues, use ppm or ppb. The unit for concentration will be identical to that of input.  
        # Define units
        lunit = "M" # M for meters
        tunit = "D" # D for days
        munit = "KG" # KG for kilograms

        #Define the active/inactive cells and their starting concentrations
        icbund = np.ones((nlay, nrow, ncol), dtype=np.int32)
        icbund[3:6, :, 9] = -1 # constant concentration/source cells

        sconc = np.zeros((nlay, nrow, ncol), dtype=np.float32)
        sconc[3:6, :, 9] =  1.0 # value of concentration (kg/m3) in layers 4-6, column 10.

        # Create the original arrays
        array1 = np.full(100, 4)
        array2 = np.zeros(100, dtype=int)
        array3 = np.arange(100)

        # Stack the arrays horizontally
        obs = np.column_stack((array1, array2, array3))

        # Define the values to filter
        values_to_filter = [3, 13, 25]

        # Filter the rows based on the third column
        filtered_obs = obs[np.isin(obs[:, 2], values_to_filter)]

        # Define the Basic Transport package (BTN)
        btn = flopy.mt3d.Mt3dBtn(mt, ncomp=1, mcomp=1, lunit=lunit, tunit=tunit, munit=munit, prsity=porosity, icbund=icbund, 
                                 sconc=sconc, ifmtcn=1, savucn=True, nprs=-1, obs=filtered_obs, nprobs=1, nprmas=1, chkmas=True, 
                                 nstp=nstp, species_names = ["Chemical_1"])

        ######################################################################################################################################################

        # Add Advection Package
        adv = flopy.mt3d.Mt3dAdv(mt, mixelm=-1, percel=0.6)

        ######################################################################################################################################################

        al = 1.0
        trpt = 0.1
        trpv = 0.1
        dmcoef = 1.0000e-5 # m/day

        # Add DSP Package
        dsp = flopy.mt3d.Mt3dDsp(mt, al=al, trpt=trpt, trpv=trpv, dmcoef=dmcoef, nocross=False)

        ######################################################################################################################################################

        # Add SSM package
        ssm = flopy.mt3d.Mt3dSsm(mt)

        ######################################################################################################################################################

        mxiter = 100
        iter1 = 30
        isolve = 3
        ncrs = 0
        accl = 1.0
        cclose = 1e-5
        iprgcg = 0

        # Add GCG package
        gcg = flopy.mt3d.Mt3dGcg(mt, mxiter=mxiter, iter1=iter1, isolve=isolve, ncrs=ncrs, accl=accl, cclose=cclose, iprgcg=iprgcg)

        ######################################################################################################################################################
       
        # Write transport model input files and run the transport model
       
        mt.write_input()
        mt.run_model()
        mt.check()
        
######################################################################################################################################################   

# If you don't understand the code, view the documentation at these links
    # FloPy: https://flopy.readthedocs.io/en/stable/index.html
    # Modflow-2005: https://water.usgs.gov/ogw/modflow/MODFLOW-2005-Guide/ground-water_flow_process.html
    # MT3DMS: https://hydro.geo.ua.edu/mt3d/index.htm
    # MT3DMS INPUT INSTRUCTIONS (use with flopy documentation): http://gmsdocs.aquaveo.com/mt3d-usgs/Input_Instructions.pdf

print("numpy version: {}".format(np.__version__))
print("matplotlib version: {}".format(mpl.__version__))
print("flopy version: {}".format(flopy.__version__))

######################################################################################################################################################

# CALIBRATION PARAMETERS

hk1 = 50  # Hydraulic conductivity (m/day) of layers 0-3 for all cells in layer
hk2 = 5   # Hydraulic conductivity (m/day) of Layers 4-7 for all cells in layer
hk3 = 0.5 # Hydraulic conductivity (m/day) of Layers 8-11 for all cells in layer
specific_yield = 0.3  # Specific yield (m3/m3), entire model domain
riv_conductance = 400 # Riverbed conductance (m2/day), for the river package (RIV)

# Call the function run_model to run the model. Transport = True (activated). Transport = False (deactivated).

run_model(hk1, hk2, hk3, specific_yield, riv_conductance, transport = False)

p:\synchronize\software\wpy64_3870\python-3.8.7.amd64\lib\site-packages\ipykernel\ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


numpy version: 1.24.4
matplotlib version: 3.7.2
flopy version: 3.4.3
FloPy is using the following executable to run the model: 2_mf2005.exe

                                  MODFLOW-2005     
    U.S. GEOLOGICAL SURVEY MODULAR FINITE-DIFFERENCE GROUND-WATER FLOW MODEL
                             Version 1.11.00 8/8/2013                        

 Using NAME file: model_flow.nam 
 Run start date and time (yyyy/mm/dd hh:mm:ss): 2023/10/12  9:46:28

 Solving:  Stress period:     1    Time step:     1    Ground-Water Flow Eqn.
 Solving:  Stress period:     1    Time step:     2    Ground-Water Flow Eqn.
 Solving:  Stress period:     1    Time step:     3    Ground-Water Flow Eqn.
 Solving:  Stress period:     1    Time step:     4    Ground-Water Flow Eqn.
 Solving:  Stress period:     1    Time step:     5    Ground-Water Flow Eqn.
 Solving:  Stress period:     1    Time step:     6    Ground-Water Flow Eqn.
 Solving:  Stress period:     1    Time step:     7    Ground-Water Flow Eqn.
 S

 Solving:  Stress period:    15    Time step:     1    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:     2    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:     3    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:     4    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:     5    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:     6    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:     7    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:     8    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:     9    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:    10    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:    11    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:    12    Ground-Water Flow Eqn.
 Solving:  Stress period:    15    Time step:    13    Ground-Wa

 Solving:  Stress period:    35    Time step:     4    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:     5    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:     6    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:     7    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:     8    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:     9    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:    10    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:    11    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:    12    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:    13    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:    14    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:    15    Ground-Water Flow Eqn.
 Solving:  Stress period:    35    Time step:    16    Ground-Wa

 Solving:  Stress period:    55    Time step:     9    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    10    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    11    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    12    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    13    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    14    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    15    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    16    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    17    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    18    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    19    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    20    Ground-Water Flow Eqn.
 Solving:  Stress period:    55    Time step:    21    Ground-Wa

...

 Solving:  Stress period:   275    Time step:     4    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:     5    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:     6    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:     7    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:     8    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:     9    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:    10    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:    11    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:    12    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:    13    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:    14    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:    15    Ground-Water Flow Eqn.
 Solving:  Stress period:   275    Time step:    16    Ground-Wa

 Solving:  Stress period:   295    Time step:     7    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:     8    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:     9    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:    10    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:    11    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:    12    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:    13    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:    14    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:    15    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:    16    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:    17    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:    18    Ground-Water Flow Eqn.
 Solving:  Stress period:   295    Time step:    19    Ground-Wa

 Solving:  Stress period:   315    Time step:     6    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:     7    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:     8    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:     9    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:    10    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:    11    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:    12    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:    13    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:    14    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:    15    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:    16    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:    17    Ground-Water Flow Eqn.
 Solving:  Stress period:   315    Time step:    18    Ground-Wa

 Solving:  Stress period:   335    Time step:    10    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    11    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    12    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    13    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    14    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    15    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    16    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    17    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    18    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    19    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    20    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    21    Ground-Water Flow Eqn.
 Solving:  Stress period:   335    Time step:    22    Ground-Wa

 Solving:  Stress period:   355    Time step:     3    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:     4    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:     5    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:     6    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:     7    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:     8    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:     9    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:    10    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:    11    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:    12    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:    13    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:    14    Ground-Water Flow Eqn.
 Solving:  Stress period:   355    Time step:    15    Ground-Wa

In [8]:
# Timeseries plot. Head vs. Column. Stress Period selectable.

# Load the data
df_head_data = pd.read_csv("5_head_data_layer5.csv")

# Define the function to plot the data for a given stress period
def plot_head_data(stress_period):
    plt.figure(figsize=(10, 6))
    plt.plot(df_head_data.columns[1:], df_head_data.iloc[stress_period, 1:], marker='o', linestyle='-')
    plt.title(f'Hydraulic Head for {df_head_data["Stress Period"][stress_period]}')
    plt.ylabel('Head')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Create an interactive widget to select the stress period
widgets.interactive(plot_head_data, stress_period=widgets.IntSlider(min=0, max=365, step=1, value=0, description='Stress Period'))


p:\synchronize\software\wpy64_3870\python-3.8.7.amd64\lib\site-packages\ipykernel\ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


interactive(children=(IntSlider(value=0, description='Stress Period', max=365), Output()), _dom_classes=('widg…

In [6]:
# Head_difference plot. Head vs. Column. Stress Period selectable.

# Load the data
df_head_data = pd.read_csv("5_head_data_layer5.csv")

# Define the function to plot the head difference data for two given stress periods
def plot_head_difference(stress_period_1, stress_period_2):
    plt.figure(figsize=(10, 6))
    
    # Calculate the difference in head values between the two stress periods
    head_difference = df_head_data.iloc[stress_period_1, 1:] - df_head_data.iloc[stress_period_2, 1:]
    
    plt.plot(df_head_data.columns[1:], head_difference, marker='o', linestyle='-')
    plt.title(f'Head Difference between {df_head_data["Stress Period"][stress_period_1]} and {df_head_data["Stress Period"][stress_period_2]}')
    plt.ylabel('Head Difference')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Create interactive widgets to select the two stress periods
widgets.interactive(plot_head_difference, 
                    stress_period_1=widgets.IntSlider(min=0, max=365, step=1, value=0, description='Stress Period'),
                    stress_period_2=widgets.IntSlider(min=0, max=365, step=1, value=0, description='Stress Period'))


p:\synchronize\software\wpy64_3870\python-3.8.7.amd64\lib\site-packages\ipykernel\ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


interactive(children=(IntSlider(value=0, description='Stress Period', max=365), IntSlider(value=0, description…